In the cell below, I have reset the environment, uninstalled libraries that would conflict with my project and replaced then with key packages needed. I also restarted the runtime to give the code a fresh start

In [ ]:

%pip -q uninstall -y numpy scipy tensorflow tensorflow-text tensorflow-decision-forests tf-keras \
  jax jaxlib xarray-einstats opencv-python opencv-python-headless opencv-contrib-python \
  albumentations albucore

%pip -q install --no-cache-dir \
  "numpy==1.26.4" \
  "scipy==1.12.0" \
  "tensorflow==2.17.0" \
  "opencv-python-headless==4.8.1.78"

import os; os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 236.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 246.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.4/601.4 MB 157.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 MB 223.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
orbax-checkpoint 0.11.23 requires jax>=0.5.0, which is not installed.
arviz 0.22.0 requires xarray-einstats>=0.3, which is not installed.
optax 0.2.5 requires jax>=0.4.27, which is not installed.
optax 0.2.5 requires jaxlib>=0.4.27, which is not installed.
tensorflow-hub 0.16.1 requires tf-keras>=2.14.1, which is not installed.
keras-hub 0.2

Double-checking to see if reset was correctly done

In [5]:
import numpy as np, tensorflow as tf, cv2, scipy

print("NumPy:", np.__version__)
print("TF:", tf.__version__)
print("OpenCV:", cv2.__version__)
print("SciPy:", scipy.__version__)



NumPy: 1.26.4
TF: 2.17.0
OpenCV: 4.8.1
SciPy: 1.12.0


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import tensorflow as tf

Loading the pretrained model into this session

In [11]:
MODEL_PATH = "/content/drive/My Drive/models/helmetdetect_CNN_Model.keras"   # change to your path
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

Preprocessing and visualizing pipeline for the CNN model

In [27]:

IMG_SIZE = (128, 128)
THRESH   = 0.30
SMOOTH_N = 8

def preprocess(frame_bgr):
    img = cv2.resize(frame_bgr, IMG_SIZE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32)/255.0
    return np.expand_dims(img, 0)

def draw_hud(frame, p_helmet, label):
    h, w = frame.shape[:2]
    cv2.rectangle(frame, (0,0), (w, 36), (0,0,0), -1)
    color = (0,0,255) if label=="HELMET" else (0,200,0)
    cv2.putText(frame, f"{label}  p_helmet={p_helmet:.2f}", (10,25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)
    bar_w = int(w * float(np.clip(p_helmet, 0, 1)))
    cv2.rectangle(frame, (10, h-20), (10+bar_w, h-10), (0,255,255), -1)
    if label == "HELMET":
        cv2.rectangle(frame, (0,0), (w-1, h-1), (0,0,255), 6)
    return frame

This code loads a video as input, runs my CNN model on every frame, applies smoothing, labels each frame as HELMET or NO-HELMET, draws a heads-up display, and saves the annotated result as a new video.

In [29]:
IN_PATH  = "/content/drive/My Drive/videos/no_helmet_test.mp4"
OUT_PATH = "/content/drive/My Drive/videos/detection_test5.mp4"


from collections import deque
cap = cv2.VideoCapture(IN_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {IN_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # fallback: 'avc1' or 'XVID'
out = cv2.VideoWriter(OUT_PATH, fourcc, fps, (w, h))

from tqdm.auto import tqdm
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None
probs = deque(maxlen=SMOOTH_N)

for _ in tqdm(range(total) if total else iter(int, 1)):
    ok, frame = cap.read()
    if not ok:
        break
    p_helmet = float(model.predict(preprocess(frame), verbose=0).ravel()[0])  # sigmoid -> P(HELMET)
    # If your model outputs P(non-HELMET), flip: p_fire = 1.0 - p_fire
    probs.append(p_helmet)
    p_avg = sum(probs)/len(probs)
    label = "HELMET" if p_avg >= THRESH else "NO-HELMET"
    vis = draw_hud(frame, p_avg, label)
    out.write(vis)

cap.release()
out.release()
print("Saved:", OUT_PATH)

  0%|          | 0/188 [00:00<?, ?it/s]

Saved: /content/drive/My Drive/videos/detection_test5.mp4
